# Explore the CMS machine-readable files (MRFs)

What price fields do the hospitals actually publish? This notebook opens the raw
MRFs (no agent, no API) and shows every available field, so we know what we can
use beyond the `low` / `high` we currently surface.

Two CMS v3.0 formats are present:

- **JSON** (Boston Children's, Lahey, South Shore): `item -> standard_charges -> payers_information`.
- **CSV "tall"** (Cambridge, Encompass): rows 1-2 are hospital metadata, row 3 is the header.

Spoiler on the common questions:

- There is **no `average`/`mean` field** in the CMS spec. But there are `minimum`,
  `maximum`, `discounted_cash`, and (CSV only) `10th_percentile` / `90th_percentile`,
  plus **one row per payer** — so we can compute mean/median/count ourselves.
- `basis` is **not** a CMS field. It is a label our pipeline added to record which
  price type a number came from: payer-negotiated dollar, else estimated/median,
  else gross chargemaster price.

In [1]:
import csv
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

csv.field_size_limit(10_000_000)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_columns", 40)

HERE = Path.cwd()
PROC = HERE if (HERE / "cms_hpt").is_dir() else HERE / "data_processing"
MRF_DIR = PROC / "cms_hpt" / "mrfs"

# filename -> real hospital name, from the manifest
manifest = pd.read_csv(MRF_DIR / "manifest.tsv", sep="\t")
NAME_OF = {r["local_file"]: r["location_name"]
           for _, r in manifest.iterrows() if isinstance(r["local_file"], str)}

files = [f for f in sorted(MRF_DIR.iterdir())
         if f.suffix.lower() in (".json", ".csv")]
overview = pd.DataFrame([{
    "file": f.name,
    "hospital": NAME_OF.get(f.name, "?"),
    "format": f.suffix.lower().lstrip("."),
    "size_mb": round(f.stat().st_size / 1e6, 1),
} for f in files])
display(overview)

,file,hospital,format,size_mb
0,04-2774441_Boston-Childrens-Longwood_StandardCharges.json,Boston Children's Longwood,json,131.9
1,042704686_lahey-hospital-medical-center-burlington_stand...,Lahey Hospital & Medical Center,json,25.3
2,042769210_south-shore-hospital_standardcharges.json,South Shore Hospital,json,36.8
3,042987822_Encompass-Health-Rehabilitation-Hospital-of-We...,Encompass Health Rehabilitation Hospital of Western Mass...,csv,7.9
4,PriceTransparency20262320.csv,Cambridge Health Alliance,csv,278.3


## 1. JSON format: schema + a sample record

Top-level metadata keys, then every field that appears under `standard_charges`
and `payers_information`, then one real item printed in full.

In [2]:
json_file = MRF_DIR / "042704686_lahey-hospital-medical-center-burlington_standardcharges.json"
with open(json_file, encoding="utf-8-sig", errors="replace") as fh:
    doc = json.load(fh)

print("Top-level keys:", list(doc.keys()))
print("hospital_name :", doc.get("hospital_name"))
print("version       :", doc.get("version"))

items = doc["standard_charge_information"]
print("n items       :", f"{len(items):,}")

sc_keys, pay_keys = set(), set()
for it in items[:30000]:
    for sc in it.get("standard_charges", []):
        sc_keys |= set(sc)
        for p in sc.get("payers_information", []) or []:
            pay_keys |= set(p)
print("\nstandard_charges fields :", sorted(sc_keys))
print("payers_information fields:", sorted(pay_keys))

# First item that actually carries payer-negotiated dollars, printed in full.
sample = next((it for it in items
               if any(sc.get("payers_information") for sc in it.get("standard_charges", []))),
              items[0])
display(Markdown("**One item with payer detail:**"))
print(json.dumps(sample, indent=2)[:2200])

Top-level keys: ['hospital_name', 'last_updated_on', 'version', 'location_name', 'hospital_address', 'license_information', 'type_2_npi', 'attestation', 'modifier_information', 'standard_charge_information']
hospital_name : Lahey Hospital & Medical Center, Burlington
version       : 3.0.0
n items       : 28,552

standard_charges fields : ['additional_generic_notes', 'billing_class', 'discounted_cash', 'gross_charge', 'maximum', 'minimum', 'modifier_code', 'payers_information', 'setting']
payers_information fields: ['10th_percentile', '90th_percentile', 'additional_payer_notes', 'count', 'median_amount', 'methodology', 'payer_name', 'plan_name', 'standard_charge_algorithm', 'standard_charge_dollar']


**One item with payer detail:**

{
  "description": "Coronary Bypass Without Ami Or Complex Principal Diagnosis",
  "code_information": [
    {
      "code": "1664",
      "type": "APR-DRG"
    }
  ],
  "standard_charges": [
    {
      "setting": "inpatient",
      "billing_class": "facility",
      "minimum": 142819.1,
      "maximum": 176218.16,
      "payers_information": [
        {
          "payer_name": "Bcbs",
          "plan_name": "All Commercial Plans",
          "methodology": "fee schedule",
          "standard_charge_dollar": 176218.16
        },
        {
          "payer_name": "Harvard Pilgrim Healthcare",
          "plan_name": "Self Insured Non Lcu All Commercial Plans",
          "methodology": "fee schedule",
          "standard_charge_dollar": 142819.1
        }
      ]
    }
  ]
}


## 2. CSV "tall" format: the full column list

Note the extra columns the JSON files don't have: `10th_percentile`,
`90th_percentile`, `standard_charge|negotiated_percentage`, `..|negotiated_algorithm`,
`..|min`, `..|max`, `..|discounted_cash`.

In [3]:
csv_file = MRF_DIR / "042987822_Encompass-Health-Rehabilitation-Hospital-of-Western-Massachusetts_standardcharges.csv"
with open(csv_file, encoding="utf-8-sig", errors="replace", newline="") as fh:
    rows = csv.reader(fh)
    meta = [next(rows) for _ in range(2)]        # hospital metadata rows
    header = next(rows)                            # real column header
    first = next(rows)                             # one data row

print("hospital metadata (row 1 labels):", meta[0][:6])
print("hospital metadata (row 2 values):", meta[1][:6])
display(pd.DataFrame({"column": header,
                      "example_value": (first + [""] * len(header))[:len(header)]}))

hospital metadata (row 1 labels): ['hospital_name', 'last_updated_on', 'version', 'location_name', 'hospital_address', 'license_number|MA']
hospital metadata (row 2 values): ['Encompass Health Rehabilitation Hospital of Western Massachusetts', '2026-01-01', '3.0.0', 'Encompass Health Rehabilitation Hospital of Western Massachusetts', '222 State Street, Ludlow, MA 01056-3437', '2912']


,column,example_value
0,description,Inpatient Rehab Per Diem Level 1
1,code|1,118; 120; 128; 138; 148; 158
2,code|1|type,RC
3,modifiers,
4,setting,inpatient
5,drug_unit_of_measurement,
6,drug_type_of_measurement,
7,standard_charge|gross,
8,standard_charge|discounted_cash,
9,payer_name,AETNA


## 3. One code across all hospitals — every field, plus computed averages

Pick a `TARGET_CODE` and pull **every raw price row** for it from all five MRFs,
keeping all the fields above. Then we compute what CMS does *not* give directly —
mean / median / count of the payer-negotiated dollars — next to the published
`gross`, `discounted_cash`, `min`, `max`, and percentiles.

(Reads the raw files each run: ~30-60s, since the JSON files must be fully loaded.)

In [4]:
TARGET_CODE = "80053"  # e.g. CPT 80053 = comprehensive metabolic panel


def _num(v):
    if v in (None, "", "-1"):
        return None
    try:
        return float(str(v).replace(",", "").replace("$", ""))
    except (ValueError, TypeError):
        return None


def scan_json(path, target):
    with open(path, encoding="utf-8-sig", errors="replace") as fh:
        doc = json.load(fh)
    out = []
    for it in doc.get("standard_charge_information", []):
        ctypes = [(c.get("code"), (c.get("type") or "").upper())
                  for c in it.get("code_information", [])]
        if not any(c == target for c, _ in ctypes):
            continue
        ctype = next((t for c, t in ctypes if c == target), None)
        for sc in it.get("standard_charges", []):
            common = {
                "code_type": ctype, "description": it.get("description"),
                "setting": sc.get("setting"), "billing_class": sc.get("billing_class"),
                "gross": _num(sc.get("gross_charge")),
                "discounted_cash": _num(sc.get("discounted_cash")),
                "min_published": _num(sc.get("minimum")), "max_published": _num(sc.get("maximum")),
                "p10": None, "p90": None,
            }
            for p in (sc.get("payers_information") or [{}]):
                out.append({**common,
                            "payer": p.get("payer_name"), "plan": p.get("plan_name"),
                            "negotiated_dollar": _num(p.get("standard_charge_dollar")),
                            "negotiated_pct": _num(p.get("standard_charge_percentage")),
                            "estimated": _num(p.get("estimated_amount")),
                            "methodology": p.get("methodology")})
    return out


def scan_csv(path, target):
    out = []
    with open(path, encoding="utf-8-sig", errors="replace", newline="") as fh:
        rows = csv.reader(fh)
        for _ in range(2):
            next(rows, None)
        header = next(rows, None) or []
        idx = {h.strip(): i for i, h in enumerate(header)}

        def g(row, *keys):
            for k in keys:
                i = idx.get(k)
                if i is not None and i < len(row):
                    return row[i].strip()
            return ""

        for row in rows:
            if not any(row):
                continue
            ctype = None
            for n in (1, 2):
                if g(row, f"code | {n}", f"code|{n}") == target:
                    ctype = g(row, f"code | {n} | type", f"code|{n}|type").upper()
                    break
            if ctype is None:
                continue
            out.append({
                "code_type": ctype, "description": g(row, "description"),
                "setting": g(row, "setting"), "billing_class": g(row, "billing_class"),
                "gross": _num(g(row, "standard_charge | gross", "standard_charge|gross")),
                "discounted_cash": _num(g(row, "standard_charge | discounted_cash",
                                          "standard_charge|discounted_cash")),
                "min_published": _num(g(row, "standard_charge | min", "standard_charge|min")),
                "max_published": _num(g(row, "standard_charge | max", "standard_charge|max")),
                "p10": _num(g(row, "10th_percentile")), "p90": _num(g(row, "90th_percentile")),
                "payer": g(row, "payer_name"), "plan": g(row, "plan_name"),
                "negotiated_dollar": _num(g(row, "standard_charge | negotiated_dollar",
                                            "standard_charge|negotiated_dollar")),
                "negotiated_pct": _num(g(row, "standard_charge | negotiated_percentage",
                                         "standard_charge|negotiated_percentage")),
                "estimated": None,
                "methodology": g(row, "standard_charge | methodology", "standard_charge|methodology"),
            })
    return out


records = []
for f in files:
    hospital = NAME_OF.get(f.name, f.name)
    rows = scan_json(f, TARGET_CODE) if f.suffix.lower() == ".json" else scan_csv(f, TARGET_CODE)
    for r in rows:
        r["hospital"] = hospital
    records.extend(rows)
    print(f"  {hospital}: {len(rows)} rows for {TARGET_CODE}")

raw = pd.DataFrame(records)
print(f"\n{len(raw)} raw price rows for code {TARGET_CODE}")
display(raw.head(15))

  Boston Children's Longwood: 48 rows for 80053
  Lahey Hospital & Medical Center: 7 rows for 80053


  South Shore Hospital: 12 rows for 80053
  Encompass Health Rehabilitation Hospital of Western Massachusetts: 2 rows for 80053


  Cambridge Health Alliance: 77 rows for 80053

146 raw price rows for code 80053


,code_type,description,setting,billing_class,gross,discounted_cash,min_published,max_published,p10,p90,payer,plan,negotiated_dollar,negotiated_pct,estimated,methodology,hospital
0,CPT,HC METABOLIC PANELCOMPREHENSIVE,both,NaN,179.0,179.0,37.12,170.05,NaN,NaN,Blue Cross,ALL PRODUCTS Inpatient,100.96,NaN,None,percent of total billed charges,Boston Children's Longwood
1,CPT,HC METABOLIC PANELCOMPREHENSIVE,both,NaN,179.0,179.0,37.12,170.05,NaN,NaN,Blue Cross,HMO/POS LW Outpatient,106.79,NaN,None,percent of total billed charges,Boston Children's Longwood
2,CPT,HC METABOLIC PANELCOMPREHENSIVE,both,NaN,179.0,179.0,37.12,170.05,NaN,NaN,Blue Cross,PPO LW Outpatient,111.82,NaN,None,percent of total billed charges,Boston Children's Longwood
3,CPT,HC METABOLIC PANELCOMPREHENSIVE,both,NaN,179.0,179.0,37.12,170.05,NaN,NaN,Blue Cross,INDEM Outpatient,118.18,NaN,None,percent of total billed charges,Boston Children's Longwood
4,CPT,HC METABOLIC PANELCOMPREHENSIVE,both,NaN,179.0,179.0,37.12,170.05,NaN,NaN,Harvard Pilgrim,LOC FOC LW Outpatient,101.08,NaN,None,percent of total billed charges,Boston Children's Longwood
5,CPT,HC METABOLIC PANELCOMPREHENSIVE,both,NaN,179.0,179.0,37.12,170.05,NaN,NaN,Harvard Pilgrim,NAT LW Outpatient,120.95,NaN,None,percent of total billed charges,Boston Children's Longwood
6,CPT,HC METABOLIC PANELCOMPREHENSIVE,both,NaN,179.0,179.0,37.12,170.05,NaN,NaN,Harvard Pilgrim,SCOAST Outpatient,90.97,NaN,None,percent of total billed charges,Boston Children's Longwood
7,CPT,HC METABOLIC PANELCOMPREHENSIVE,both,NaN,179.0,179.0,37.12,170.05,NaN,NaN,Aetna,COMM Inpatient/Outpatient,150.06,NaN,None,percent of total billed charges,Boston Children's Longwood
8,CPT,HC METABOLIC PANELCOMPREHENSIVE,both,NaN,179.0,179.0,37.12,170.05,NaN,NaN,Cigna,ALL PRODUCTS Inpatient,157.54,NaN,None,percent of total billed charges,Boston Children's Longwood
9,CPT,HC METABOLIC PANELCOMPREHENSIVE,both,NaN,179.0,179.0,37.12,170.05,NaN,NaN,Cigna,ALL PRODUCTS Outpatient,161.53,NaN,None,percent of total billed charges,Boston Children's Longwood


### Per-hospital summary — including the average/median we compute ourselves

`n_payers`, `mean`, `median`, `min`, `max` are derived from the individual
payer-negotiated dollars. `gross`, `discounted_cash`, `min_published`,
`max_published`, `p10`, `p90` are taken straight from the file.

In [5]:
def summarize(g):
    neg = g["negotiated_dollar"].dropna()
    return pd.Series({
        "code_type": g["code_type"].dropna().iloc[0] if g["code_type"].notna().any() else None,
        "n_payer_rows": len(g),
        "n_negotiated": int(neg.shape[0]),
        "neg_mean": round(neg.mean(), 2) if len(neg) else None,
        "neg_median": round(neg.median(), 2) if len(neg) else None,
        "neg_min": neg.min() if len(neg) else None,
        "neg_max": neg.max() if len(neg) else None,
        "gross": g["gross"].dropna().max(),
        "discounted_cash": g["discounted_cash"].dropna().max(),
        "min_published": g["min_published"].dropna().min(),
        "max_published": g["max_published"].dropna().max(),
        "p10": g["p10"].dropna().min(),
        "p90": g["p90"].dropna().max(),
    })


if len(raw):
    summary = raw.groupby("hospital").apply(summarize, include_groups=False)
    display(Markdown(f"**Code {TARGET_CODE} — per-hospital price summary**"))
    display(summary)
else:
    print(f"No hospital publishes code {TARGET_CODE}.")

**Code 80053 — per-hospital price summary**

,code_type,n_payer_rows,n_negotiated,neg_mean,neg_median,neg_min,neg_max,gross,discounted_cash,min_published,max_published,p10,p90
hospital,,,,,,,,,,,,,
Boston Children's Longwood,CPT,48,42,129.62,133.29,41.8200,170.050,179.0,179.0,37.1200,170.050,NaN,NaN
Cambridge Health Alliance,CPT,77,77,181.36,320.00,9.3100,320.000,320.0,320.0,9.3100,320.000,0.04,320.0
Encompass Health Rehabilitation Hospital of Western Massachusetts,CDM,2,2,33.41,33.41,32.7744,34.048,64.0,48.0,32.7744,34.048,NaN,NaN
Lahey Hospital & Medical Center,CDM,7,6,24.06,22.92,10.5600,38.600,127.0,127.0,10.5600,38.600,NaN,NaN
South Shore Hospital,HCPCS,12,11,19.54,20.62,10.5600,27.370,153.0,107.1,10.5600,27.370,NaN,NaN
